# Unbiased Vergleich: MixUp-Regressor vs. Synthetischer LLM-Regressor

In diesem Notebook vergleichen wir die beiden Regressionsmodelle zur Komplexitätsmessung unter fairen und unvoreingenommenen Bedingungen:

1. **LLM-Synthetisiertes Stufen-Evaluationsset:** `lebenshilfe_dataset_with_steps.json` (245 Samples). Das synthetische Modell wurde nach einem ähnlichen Prinzip trainiert (LLM-generierte Stufen). Dies führt zu einem Bias zugunsten des synthetischen Modells.
2. **Sentence-MixUp Evaluationsset:** Koppelung von originalen Lebenshilfe-Satzpaaren (LS und AS) zu zufälligen Mischungsverhältnissen (490 Samples). Da das MixUp-Modell auf solchen Satzmischungen trainiert wurde, bietet dieses Dataset faire/komplementäre Evaluationsbedingungen.

Beide Modelle werden auf beiden Datensätzen evaluiert, um den Bias-Effekt zu messen.

In [ ]:
import json
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import spacy
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import Dataset, DataLoader
from scipy.stats import pearsonr, spearmanr

# Set Style
sns.set_theme(style="whitegrid")

# Pfade relativ zum Notebook-Verzeichnis
LH_WITH_STEPS_PATH = "../../results/lebenshilfe_dataset_with_steps.json"
LH_NO_PARAGRAPHS_PATH = "../../results/lebenshilfe_dataset_no_paragraphs.json"
MIXUP_MODEL_PATH = "../../results/bilstm_mixup_regression_hybrid_cyclic.pt"
MIXUP_VOCAB_PATH = "../../results/mixup_vocab.json"
SYNTHETIC_MODEL_PATH = "../../results/bilstm_synthetic_regression.pt"
SYNTHETIC_VOCAB_PATH = "../../results/synthetic_vocab.json"
IMG_DIR = "../../research/img/analysis"

os.makedirs(IMG_DIR, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print(f"Nutze Device: {device}")

## 1. Modellarchitektur & Setup

In [ ]:
class BiLSTMRegressor(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128, dropout=0.3):
        super(BiLSTMRegressor, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, 1)
        self.dropout = nn.Dropout(dropout)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        _, (hidden, _) = self.lstm(embedded)
        hidden = torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)
        out = self.fc(self.dropout(hidden))
        return self.sigmoid(out).squeeze(-1)

# Spacy für die Tokenisierung / Satzsegmentierung
nlp = spacy.load("de_core_news_sm", disable=["ner", "tagger", "lemmatizer"])

def load_vocab_dict(vocab_path):
    with open(vocab_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if "stoi" in data:
        return data["stoi"]
    return data

class EvalDataset(Dataset):
    def __init__(self, samples, vocab_map, nlp_spacy, max_len=256):
        self.samples = samples
        self.vocab = vocab_map
        self.nlp = nlp_spacy
        self.max_len = max_len
        self.unk_id = self.vocab.get("<unk>") or self.vocab.get("<UNK>") or 1
        
    def __len__(self):
        return len(self.samples)
        
    def __getitem__(self, idx):
        item = self.samples[idx]
        doc = self.nlp(item["text"])
        tokens = [t.text.lower() for t in doc if not t.is_space][:self.max_len]
        token_ids = [self.vocab.get(t, self.unk_id) for t in tokens]
        if not token_ids:
            token_ids = [0]
        return torch.tensor(token_ids, dtype=torch.long), torch.tensor(item["target"], dtype=torch.float32)

def pad_collate_fn(batch):
    sequences, targets = zip(*batch)
    padded_seqs = torch.nn.utils.rnn.pad_sequence(sequences, batch_first=True, padding_value=0)
    targets = torch.tensor(targets, dtype=torch.float32)
    return padded_seqs, targets

def evaluate_model(model_path, vocab_path, samples):
    vocab_map = load_vocab_dict(vocab_path)
    vocab_size = len(vocab_map)
    
    model = BiLSTMRegressor(vocab_size=vocab_size).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    
    ds = EvalDataset(samples, vocab_map, nlp)
    loader = DataLoader(ds, batch_size=32, shuffle=False, collate_fn=pad_collate_fn)
    
    preds, targets = [], []
    with torch.no_grad():
        for x_b, y_b in loader:
            x_b = x_b.to(device)
            out = model(x_b).cpu().numpy()
            preds.extend(out)
            targets.extend(y_b.numpy())
            
    return np.array(preds), np.array(targets)

## 2. Dataset 1: LLM-Synthetisiertes Stufen-Evaluationsset laden

In [ ]:
with open(LH_WITH_STEPS_PATH, "r", encoding="utf-8") as f:
    articles = json.load(f)
    
synthetic_lh_samples = []
for art in articles:
    ls_text = art.get("ls_text", "").strip()
    as_text = art.get("as_text", "").strip()
    steps = art.get("intermediate_steps", {})
    
    if ls_text:
        synthetic_lh_samples.append({"text": ls_text, "target": 0.0})
    if as_text:
        synthetic_lh_samples.append({"text": as_text, "target": 1.0})
        
    for step_str, step_text in steps.items():
        try:
            target_val = float(step_str)
            if step_text and step_text.strip():
                synthetic_lh_samples.append({
                    "text": step_text.strip(),
                    "target": target_val
                })
        except ValueError:
            continue

print(f"Geladene LLM-Stufen-Samples: {len(synthetic_lh_samples)}")

## 3. Dataset 2: Sentence-MixUp Evaluationsset generieren

In [ ]:
with open(LH_NO_PARAGRAPHS_PATH, "r", encoding="utf-8") as f:
    lh_articles = json.load(f)

random.seed(99)
mixup_lh_samples = []
mixtures_per_pair = 10

for row in lh_articles:
    ls_doc = nlp(str(row["ls_text"]))
    as_doc = nlp(str(row["as_text"]))
    
    ls_sents = []
    for sent in ls_doc.sents:
        text = sent.text.strip()
        tokens = [t.text.lower() for t in sent if not t.is_space]
        if len(tokens) > 0:
            ls_sents.append((tokens, len(text)))
            
    as_sents = []
    for sent in as_doc.sents:
        text = sent.text.strip()
        tokens = [t.text.lower() for t in sent if not t.is_space]
        if len(tokens) > 0:
            as_sents.append((tokens, len(text)))
            
    num_leicht = len(ls_sents)
    num_alltag = len(as_sents)
    
    if num_leicht == 0 or num_alltag == 0:
        continue
        
    for _ in range(mixtures_per_pair):
        start_l, end_l = sorted([random.randint(0, num_leicht), random.randint(0, num_leicht)])
        sample_l = ls_sents[start_l:end_l]
        
        start_a, end_a = sorted([random.randint(0, num_alltag), random.randint(0, num_alltag)])
        sample_a = as_sents[start_a:end_a]
        
        if len(sample_l) == 0 and len(sample_a) == 0:
            continue
            
        char_len_l = sum(item[1] for item in sample_l)
        char_len_a = sum(item[1] for item in sample_a)
        total_char_len = char_len_l + char_len_a
        regression_target = char_len_l / total_char_len if total_char_len > 0 else 0.5
        
        # Harmonisiere Komplexität: 1.0 - target (da MixUp 1.0=LS und 0.0=AS target hatte)
        complexity_target = 1.0 - regression_target
        
        mixed_sentences = [item[0] for item in sample_l] + [item[0] for item in sample_a]
        random.shuffle(mixed_sentences)
        
        mixed_text = " ".join([" ".join(tokens) for tokens in mixed_sentences])
        
        mixup_lh_samples.append({
            "text": mixed_text,
            "target": complexity_target
        })

print(f"Generierte Sentence-MixUp-Samples: {len(mixup_lh_samples)}")

## 4. Evaluationen durchführen und vergleichen

In [ ]:
def get_metrics(preds, targets):
    mse = np.mean((preds - targets) ** 2)
    mae = np.mean(np.abs(preds - targets))
    r, _ = pearsonr(preds, targets)
    rho, _ = spearmanr(preds, targets)
    return mse, mae, r, rho

results = {}

for dataset_name, samples in [("LLM-Stufen", synthetic_lh_samples), ("Sentence-MixUp", mixup_lh_samples)]:
    print(f"\nEvaluating on {dataset_name}...")
    
    # MixUp Evaluieren
    raw_preds_mixup, targets_eval = evaluate_model(MIXUP_MODEL_PATH, MIXUP_VOCAB_PATH, samples)
    preds_mixup = 1.0 - raw_preds_mixup
    
    # Synthetisches Modell Evaluieren
    preds_synthetic, _ = evaluate_model(SYNTHETIC_MODEL_PATH, SYNTHETIC_VOCAB_PATH, samples)
    
    # Metriken berechnen
    mse_m, mae_m, r_m, rho_m = get_metrics(preds_mixup, targets_eval)
    mse_s, mae_s, r_s, rho_s = get_metrics(preds_synthetic, targets_eval)
    
    # Speichern
    results[dataset_name] = {
        "mixup": {"mse": mse_m, "mae": mae_m, "r": r_m, "rho": rho_m},
        "synthetic": {"mse": mse_s, "mae": mae_s, "r": r_s, "rho": rho_s}
    }
    
    # Ausgeben
    print("=" * 70)
    print(f"METRIKEN FÜR {dataset_name.upper()}")
    print("=" * 70)
    print(f"Metrik        MixUp-Modell (Variante D)      Synthetisches LLM-Modell")
    print(f"MSE           {mse_m:.4f}                          {mse_s:.4f}")
    print(f"MAE           {mae_m:.4f}                          {mae_s:.4f}")
    print(f"Pearson r     {r_m:.4f}                          {r_s:.4f}")
    print(f"Spearman rho  {rho_m:.4f}                          {rho_s:.4f}")
    print("=" * 70)
    
    filename_base = "synthetic_lh_dataset" if "stufen" in dataset_name.lower() else "mixup_lh_dataset"
    
    # 1. Boxplot plotten & speichern
    df_comp = pd.DataFrame({
        "Zielstufe": [f"{t:.2f}" for t in targets_eval] * 2 if "stufen" in dataset_name.lower() else [f"{np.round(t*4)/4:.2f}" for t in targets_eval] * 2,
        "Vorhergesagter Score": np.concatenate([preds_mixup, preds_synthetic]),
        "Modell": ["MixUp-Modell (Variante D)"] * len(preds_mixup) + ["Synthetisches LLM-Modell"] * len(preds_synthetic)
    })
    
    plt.figure(figsize=(12, 6))
    sns.boxplot(data=df_comp, x="Zielstufe", y="Vorhergesagter Score", hue="Modell", palette="Set2")
    plt.title(f"Boxplot-Vergleich auf dem {dataset_name} Dataset")
    plt.xlabel("Zielstufe (Complexity Ground Truth)")
    plt.ylabel("Vorhergesagter Komplexitäts-Score")
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.legend(title="Modell")
    plt.tight_layout()
    plt.savefig(f"{IMG_DIR}/{filename_base}_boxplot.png", dpi=300)
    plt.show()
    
    # 2. Regressionplot plotten & speichern
    plt.figure(figsize=(10, 6))
    sns.regplot(x=targets_eval, y=preds_mixup, label=f"MixUp (r={r_m:.3f})", scatter_kws={"alpha": 0.3}, line_kws={"color": "red"})
    sns.regplot(x=targets_eval, y=preds_synthetic, label=f"Synthetisch (r={r_s:.3f})", scatter_kws={"alpha": 0.3}, line_kws={"color": "blue"})
    plt.plot([0, 1], [0, 1], "k--", label="Ideale Monotonie (1:1)")
    plt.title(f"Regressions-Vergleich auf dem {dataset_name} Dataset")
    plt.xlabel("Zielstufe (Complexity Ground Truth)")
    plt.ylabel("Vorhergesagter Komplexitäts-Score")
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.savefig(f"{IMG_DIR}/{filename_base}_regplot.png", dpi=300)
    plt.show()
    
    # 3. Advanced KDE Density Comparison
    plt.figure(figsize=(10, 5))
    sns.kdeplot(targets_eval, label="Target (Ground Truth)", fill=True, alpha=0.15, color="gray", linewidth=2)
    sns.kdeplot(preds_mixup, label=f"MixUp (r={r_m:.3f})", color="red", linewidth=2)
    sns.kdeplot(preds_synthetic, label=f"Synthetic (r={r_s:.3f})", color="blue", linewidth=2)
    plt.title(f"Dichtevergleich (KDE) der Vorhersagen vs. Ground Truth - {dataset_name}")
    plt.xlabel("Komplexitäts-Score (0.0 = LS, 1.0 = AS)")
    plt.ylabel("Dichte")
    plt.xlim(-0.1, 1.1)
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"{IMG_DIR}/{filename_base}_density_comparison.png", dpi=300)
    plt.show()
    
    # 4. Advanced Residuals Distribution
    res_mixup = preds_mixup - targets_eval
    res_synth = preds_synthetic - targets_eval
    plt.figure(figsize=(10, 5))
    sns.kdeplot(res_mixup, label=f"MixUp (Bias: {np.mean(res_mixup):.3f}, SD: {np.std(res_mixup):.3f})", color="red", fill=True, alpha=0.2)
    sns.kdeplot(res_synth, label=f"Synthetic (Bias: {np.mean(res_synth):.3f}, SD: {np.std(res_synth):.3f})", color="blue", fill=True, alpha=0.2)
    plt.axvline(0, color="black", linestyle="--", alpha=0.7)
    plt.title(f"Verteilung der Residuen (Fehler: Pred - Target) - {dataset_name}")
    plt.xlabel("Fehler (Residuum)")
    plt.ylabel("Dichte")
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"{IMG_DIR}/{filename_base}_residuals_distribution.png", dpi=300)
    plt.show()
    
    # 5. Advanced MAE by Target Complexity Bin
    bins = [0.0, 0.2, 0.4, 0.6, 0.8, 1.01]
    bin_labels = ["0.0-0.2 (Einfach)", "0.2-0.4", "0.4-0.6 (Mittel)", "0.6-0.8", "0.8-1.0 (Schwer)"]
    df_bins = pd.DataFrame({
        "target": targets_eval,
        "err_mixup": np.abs(preds_mixup - targets_eval),
        "err_synth": np.abs(preds_synthetic - targets_eval)
    })
    df_bins["bin"] = pd.cut(df_bins["target"], bins=bins, labels=bin_labels, include_lowest=True)
    mae_bin_m = df_bins.groupby("bin", observed=False)["err_mixup"].mean()
    mae_bin_s = df_bins.groupby("bin", observed=False)["err_synth"].mean()
    
    df_plot_mae = pd.DataFrame({
        "Komplexitätsbereich": bin_labels * 2,
        "MAE": np.concatenate([mae_bin_m.values, mae_bin_s.values]),
        "Modell": ["MixUp-Modell (Variante D)"] * len(bin_labels) + ["Synthetisches LLM-Modell"] * len(bin_labels)
    })
    plt.figure(figsize=(12, 5))
    sns.barplot(data=df_plot_mae, x="Komplexitätsbereich", y="MAE", hue="Modell", palette="Set2")
    plt.title(f"Mittlerer Absoluter Fehler (MAE) nach Komplexitätsbereich - {dataset_name}")
    plt.ylabel("MAE")
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.savefig(f"{IMG_DIR}/{filename_base}_mae_by_bin.png", dpi=300)
    plt.show()